## **X5**: TT - Pipeline Audit
* October 7°, 2026
#### ESCOM - IPN:

#### *B.S. in Computer Science*
> Miguel Alexander Sanchez Garcia

#### **0° Introduction**

Modules 6 and 7 and experiment X4 found no quantum advantage. Part of the literature reviewed in the report describes quantum models that outperform classical ones on medical data, so before accepting the result it is worth asking whether something in the pipeline is wrong in a way that could hide an advantage.

This notebook checks, with implementations **independent** of the ones used in the modules, the places where such an error could live:

1. **Labels and splits.** They are rebuilt from the four original CSV files of the CBIS-DDSM, without going through Module 1.
2. **Leakage.** The angular scaling of Module 4 is refitted on train only, and the test angles must come out identical.
3. **The quantum kernel.** It is computed with `FidelityQuantumKernel`, which builds a compute-uncompute circuit per pair, a different code path from the statevector shortcut of Modules 5 and 6.
4. **The QSVM.** The `QSVC` class of `qiskit-machine-learning` is compared with the precomputed-kernel SVM of X4.
5. **The encoding.** Two checks:
   - **The mapping from features to qubits.** The angle $x_i$ must act on qubit $i$.
   - **Invariance under reordering.** With full entanglement every pair of qubits is coupled, so reordering the features can only relabel the qubits and must leave the kernel unchanged.
6. **The classical references.** The same classifier has to give comparable results on the same information.

No check here changes data or results.

#### **1° Setup**

In [1]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import qiskit
import qiskit_machine_learning
from qiskit.circuit.library import pauli_feature_map, zz_feature_map
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.kernels import FidelityQuantumKernel, FidelityStatevectorKernel
from sklearn.preprocessing import QuantileTransformer
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
SEED = 42
FINDINGS = ['mass', 'calcification']
X_COLUMNS = [f'x_{i:02d}' for i in range(1, 13)]
ROOT = Path('/Users/alexander_san/TT')
DATA, M3_DIR, M4_DIR = ROOT / 'Data', ROOT / 'Data/processed/m3', ROOT / 'Data/processed/m4'
RESULTS_DIR = ROOT / 'Code/results'
maps = {'C4': zz_feature_map(12, reps=1), 'C5': pauli_feature_map(12, reps=1, paulis=['Y', 'ZZ'])}
checks = []


def check(area, name, value, passed):
    checks.append({'area': area, 'check': name, 'value': float(value), 'passed': bool(passed)})


frames = {ft: pd.read_parquet(M4_DIR / f'x_{ft}.parquet') for ft in FINDINGS}
print(f'qiskit {qiskit.__version__} | qiskit-machine-learning {qiskit_machine_learning.__version__}')
for ft, f in frames.items():
    print(f'{ft:13s}: {len(f)} lesions in Module 4')

qiskit 2.3.0 | qiskit-machine-learning 0.9.0
mass         : 1696 lesions in Module 4
calcification: 1872 lesions in Module 4


#### **2° Labels and splits from the original CSV files**

**a.** The case identifier of the pipeline is `<Mass|Calc>-<Training|Test>_<patient>_<breast>_<view>_<abnormality id>`. It is rebuilt here from the columns of each CSV, the label is defined as malignant = 1 and everything else = 0, and the split is read from the file name.

In [2]:
raw = []
for kind, prefix in [('mass', 'Mass'), ('calc', 'Calc')]:
    for split, tag in [('train', 'Training'), ('test', 'Test')]:
        csv = pd.read_csv(DATA / f'{kind}_case_description_{split}_set.csv')
        case_id = (prefix + '-' + tag + '_' + csv['patient_id'] + '_' + csv['left or right breast'] + '_'
                   + csv['image view'] + '_' + csv['abnormality id'].astype(str))
        raw.append(pd.DataFrame({'case_id': case_id, 'finding_type': 'mass' if kind == 'mass' else 'calcification',
                                 'split_csv': split, 'pathology_csv': csv['pathology'],
                                 'label_csv': (csv['pathology'] == 'MALIGNANT').astype(int)}))
raw = pd.concat(raw, ignore_index=True)
print(f'rows in the four CSV files: {len(raw)}; distinct case identifiers: {raw.case_id.nunique()}')
print(raw.pathology_csv.value_counts().to_string())
for ft in FINDINGS:
    merged = frames[ft].merge(raw, on=['case_id', 'finding_type'], how='left')
    missing = int(merged.label_csv.isna().sum())
    check('labels', f'{ft}: every lesion of Module 4 exists in the original CSV files', missing, missing == 0)
    wrong_label = int((merged.label != merged.label_csv).sum())
    check('labels', f'{ft}: labels equal MALIGNANT = 1 from the CSV files', wrong_label, wrong_label == 0)
    wrong_split = int((merged.split != merged.split_csv).sum())
    check('labels', f'{ft}: train/test split equals the file of origin', wrong_split, wrong_split == 0)
    counts = merged.groupby(['split', 'label']).size().to_dict()
    print(f'{ft:13s}: {counts}')
expected = {'mass': (912, 784), 'calcification': (1199, 673)}
for ft, (b, m) in expected.items():
    got = frames[ft].label.value_counts().to_dict()
    check('labels', f'{ft}: class counts equal the EDA ({b} benign, {m} malignant)', 0, got.get(0) == b and got.get(1) == m)
display(pd.DataFrame(checks))

rows in the four CSV files: 3568; distinct case identifiers: 3568
pathology_csv
MALIGNANT                  1457
BENIGN                     1429
BENIGN_WITHOUT_CALLBACK     682
mass         : {('test', 0): 231, ('test', 1): 147, ('train', 0): 681, ('train', 1): 637}
calcification: {('test', 0): 197, ('test', 1): 129, ('train', 0): 1002, ('train', 1): 544}


,area,check,value,passed
0,labels,mass: every lesion of Module 4 exists in the o...,0.0,True
1,labels,mass: labels equal MALIGNANT = 1 from the CSV ...,0.0,True
2,labels,mass: train/test split equals the file of origin,0.0,True
3,labels,calcification: every lesion of Module 4 exists...,0.0,True
4,labels,calcification: labels equal MALIGNANT = 1 from...,0.0,True
5,labels,calcification: train/test split equals the fil...,0.0,True
6,labels,"mass: class counts equal the EDA (912 benign, ...",0.0,True
7,labels,calcification: class counts equal the EDA (119...,0.0,True


#### **3° Leakage**

**a.** Refit the selection-and-scaling step of Module 4 on train only and transform the test lesions. If Module 4 had used any test information, the angles would differ.

In [3]:
for ft in FINDINGS:
    meta = json.loads((M4_DIR / f'x_{ft}.json').read_text())
    m3 = pd.read_parquet(M3_DIR / f'features_{ft}_raw.parquet').set_index('case_id').loc[frames[ft].case_id]
    train = frames[ft].split.eq('train').to_numpy()
    qt = QuantileTransformer(n_quantiles=min(1000, int(train.sum())), output_distribution='uniform',
                             subsample=10**9, random_state=SEED).fit(m3[meta['features']].values[train])
    angles = np.pi * qt.transform(m3[meta['features']].values)
    err = np.abs(angles - frames[ft][X_COLUMNS].to_numpy()).max()
    check('leakage', f'{ft}: angles refitted on train only equal Module 4, test included', err, err < 1e-12)
    out = int(((frames[ft][X_COLUMNS] < 0) | (frames[ft][X_COLUMNS] > np.pi + 1e-12)).to_numpy().sum())
    check('leakage', f'{ft}: every angle lies in [0, pi], test included', out, out == 0)
    split_patients = int((frames[ft][train].groupby('patient_id').fold.nunique() > 1).sum())
    shared = len(set(frames[ft][train].patient_id) & set(frames[ft][~train].patient_id))
    check('leakage', f'{ft}: no patient split across folds nor across train and test', split_patients + shared, split_patients + shared == 0)
display(pd.DataFrame(checks).tail(6))

,area,check,value,passed
8,leakage,mass: angles refitted on train only equal Modu...,0.0,True
9,leakage,"mass: every angle lies in [0, pi], test included",0.0,True
10,leakage,mass: no patient split across folds nor across...,0.0,True
11,leakage,calcification: angles refitted on train only e...,0.0,True
12,leakage,"calcification: every angle lies in [0, pi], te...",0.0,True
13,leakage,calcification: no patient split across folds n...,0.0,True


#### **4° The quantum kernel through an independent code path**

**a.** `FidelityQuantumKernel` builds, for each pair, the circuit $U(\mathbf{x}')^\dagger U(\mathbf{x})$ and reads the probability of the all-zeros outcome; with its default settings that probability is exact (H-032). It shares no code with the statevector inner products used in Modules 5 and 6 and in X4.

In [4]:
rng = np.random.default_rng(SEED)
for ft in FINDINGS:
    X = frames[ft][X_COLUMNS].to_numpy(float)
    idx = rng.choice(len(X), 12, replace=False)
    for q, fm in maps.items():
        t0 = time.perf_counter()
        circuit_kernel = FidelityQuantumKernel(feature_map=fm).evaluate(X[idx])
        sv_kernel = FidelityStatevectorKernel(feature_map=fm).evaluate(X[idx])
        err = np.abs(circuit_kernel - sv_kernel).max()
        check('quantum kernel', f'{ft}: {q} compute-uncompute kernel equals statevector kernel (12 x 12)', err, err < 1e-9)
        print(f'{ft:13s} {q}: max difference {err:.2e} ({time.perf_counter() - t0:.0f} s)')
display(pd.DataFrame(checks).tail(4))

mass          C4: max difference 4.23e-11 (8 s)


mass          C5: max difference 9.40e-13 (8 s)


calcification C4: max difference 5.74e-11 (9 s)


calcification C5: max difference 3.22e-11 (9 s)


,area,check,value,passed
14,quantum kernel,mass: C4 compute-uncompute kernel equals state...,4.232538e-11,True
15,quantum kernel,mass: C5 compute-uncompute kernel equals state...,9.396155e-13,True
16,quantum kernel,calcification: C4 compute-uncompute kernel equ...,5.742108e-11,True
17,quantum kernel,calcification: C5 compute-uncompute kernel equ...,3.215533e-11,True


**b.** The `QSVC` class of `qiskit-machine-learning` against the SVM with a precomputed kernel used in X4, on 120 training and 40 test lesions. Both must give the same decision function.

In [5]:
for ft in FINDINGS:
    f = frames[ft]
    tr_idx = rng.choice(np.flatnonzero(f.split.eq('train')), 120, replace=False)
    te_idx = rng.choice(np.flatnonzero(f.split.eq('test')), 40, replace=False)
    X, y = f[X_COLUMNS].to_numpy(float), f.label.to_numpy(int)
    kernel = FidelityStatevectorKernel(feature_map=maps['C4'])
    qsvc = QSVC(quantum_kernel=kernel, C=1.0, class_weight='balanced').fit(X[tr_idx], y[tr_idx])
    K_tr, K_te = kernel.evaluate(X[tr_idx]), kernel.evaluate(X[te_idx], X[tr_idx])
    svc = SVC(kernel='precomputed', C=1.0, class_weight='balanced').fit(K_tr, y[tr_idx])
    err = np.abs(qsvc.decision_function(X[te_idx]) - svc.decision_function(K_te)).max()
    check('QSVM', f'{ft}: QSVC of qiskit-machine-learning equals the precomputed-kernel SVM of X4', err, err < 1e-8)
    print(f'{ft:13s}: max difference in decision function {err:.2e}')

mass         : max difference in decision function 0.00e+00


calcification: max difference in decision function 0.00e+00


#### **5° The encoding**

**a.** Each angle must act on its own qubit. With C5 and no ansatz, the local Z value of qubit $i$ is exactly $-\sin(2x_i)$ (2°a of 5a), so changing only $x_i$ must change only $\langle Z_i\rangle$, and in that way.

In [6]:
x = frames['mass'][X_COLUMNS].to_numpy(float)[0]
fm = maps['C5']
z_of = lambda v: np.array([Statevector(fm.assign_parameters(dict(zip(fm.parameters, v)))).expectation_value(
    SparsePauliOp('I' * (11 - i) + 'Z' + 'I' * i)).real for i in range(12)])
base = z_of(x)
worst = 0.0
for i in range(12):
    v = x.copy()
    v[i] = (v[i] + 0.7) % np.pi
    moved = z_of(v)
    others = np.delete(np.abs(moved - base), i).max()
    worst = max(worst, others, abs(moved[i] + np.sin(2 * v[i])))
check('encoding', 'changing x_i changes only <Z_i> of C5, to -sin(2 x_i)', worst, worst < 1e-12)
print(f'largest deviation over the 12 features: {worst:.2e}')

largest deviation over the 12 features: 6.11e-15


**b.** With full entanglement, every pair of qubits receives a ZZ term with the same functional form. Reordering the features therefore only relabels the qubits, and the fidelity between two lesions cannot change. This is a property of the encoding, checked here numerically.

In [7]:
X = frames['calcification'][X_COLUMNS].to_numpy(float)[:30]
perm = np.random.default_rng(SEED).permutation(12)
for q, fm in maps.items():
    K = FidelityStatevectorKernel(feature_map=fm).evaluate(X)
    K_perm = FidelityStatevectorKernel(feature_map=fm).evaluate(X[:, perm])
    err = np.abs(K - K_perm).max()
    check('encoding', f'{q}: the kernel is invariant under a reordering of the features', err, err < 1e-10)
    print(f'{q}: max change after permuting the features {err:.2e}')

C4: max change after permuting the features 1.11e-16


C5: max change after permuting the features 4.72e-16


#### **6° Classical references**

**a.** The same information through different classical models should give comparable results. Three models on the 12 angles:

- the MLP of Module 7 on C1;
- the RBF SVM of X4;
- the logistic regression of X1, which uses the raw features.

Their cross-validated AUCs must agree within the noise of the folds. A gross disagreement would point to an error in one of the pipelines.

In [8]:
m7 = pd.read_csv(RESULTS_DIR / '7_cv_por_fold.csv')
x4 = pd.read_csv(RESULTS_DIR / 'X4_cv_por_fold.csv')
x1 = pd.read_csv(RESULTS_DIR / 'X1_auc_por_condicion.csv')
rows = []
for ft in FINDINGS:
    mlp = m7[(m7.finding_type == ft) & (m7.analysis == 'main') & (m7.condition == 'C1')].auc
    svm = x4[(x4.finding_type == ft) & (x4.model == 'SVM RBF (D-027)')].auc
    lr = x1[(x1.finding_type == ft) & (x1.model == 'LR') & (x1.feature_set == '12 of Module 4 (fixed)')].auc.iloc[0]
    rows.append({'finding_type': ft, 'MLP on C1 (M7)': mlp.mean(), 'SVM RBF (X4)': svm.mean(), 'LR (X1)': lr, 'fold sd of the MLP': mlp.std()})
    spread = max(mlp.mean(), svm.mean(), lr) - min(mlp.mean(), svm.mean(), lr)
    check('classical references', f'{ft}: three classical models agree within one fold standard deviation', spread, spread < mlp.std())
display(pd.DataFrame(rows).round(3))

,finding_type,MLP on C1 (M7),SVM RBF (X4),LR (X1),fold sd of the MLP
0,mass,0.652,0.674,0.650,0.068
1,calcification,0.742,0.746,0.758,0.038


#### **7° Summary**

In [9]:
audit = pd.DataFrame(checks)
audit.to_csv(RESULTS_DIR / 'X5_auditoria.csv', index=False)
display(audit.groupby('area').passed.agg(['sum', 'count']))
print(f'Checks passed: {int(audit.passed.sum())}/{len(audit)}')

,sum,count
area,,
QSVM,2,2
classical references,2,2
encoding,3,3
labels,8,8
leakage,6,6
quantum kernel,4,4


Checks passed: 25/25


#### **8° Consequence**

**No error was found that could hide an advantage. All 25 checks pass.**

| Area | What was checked independently | Result |
|---|---|---|
| Labels and splits | Rebuilt from the four original CSV files without Module 1: 3,568 distinct cases, malignant = 1, split by file of origin | Identical for every lesion; class counts equal the EDA |
| Leakage | Module 4 refitted on train only and applied to test | Angles identical to $10^{-12}$; no patient split across folds or across train and test |
| Quantum kernel | `FidelityQuantumKernel`, which builds a compute-uncompute circuit per pair | Equal to the statevector kernel to $6\times10^{-11}$ |
| QSVM | The `QSVC` class of `qiskit-machine-learning` against the SVM of X4 | Decision functions identical |
| Encoding | Each angle acts on its own qubit; reordering the features leaves the full-entanglement kernel unchanged | Both to $10^{-14}$ |
| Classical references | MLP (M7), SVM (X4) and logistic regression (X1) on the same information | They agree within the fold noise |

**What the audit cannot rule out** are choices of design, not errors:
- the encoding (ZZ, full entanglement, angles in $[0,\pi]$);
- the readout of Architecture B;
- the univariate selection of 12 features.

Experiments X4 and X6 vary exactly those choices.

**How to read reports of advantage.** Huang et al. (2021, verified quotation in H-031) found that *"a variety of common quantum models in the literature perform similarly or worse than classical ML"*, because of the small geometric difference. When a paper reports that a quantum model outperforms a classical one, four questions decide whether the result contradicts this one:

1. Was the classical baseline given the same tuning and the same features?
2. Is the difference larger than its uncertainty, with an interval or a test, and with patients rather than images as the unit?
3. Was the test split used once, after every choice?
4. Is the quantum kernel actually hard to compute classically? Amplitude encoding, for example, gives a classical polynomial kernel (X4).

This notebook does not answer those questions for the papers cited in the report. They should be checked paper by paper before §2 and §7 are written.
